In [ ]:
import pandas as pd # manipulation des tableaux
import numpy as np # calcul matriciel, algebrique
import matplotlib.pyplot as plt # visualisation graphique des donnees, pyplot pour le tracage des graphiques en 2D
import seaborn as sns # manipulation des statistiques

In [ ]:
import warnings # generer les erreurs et les alertes
warnings.filterwarnings("ignore") # ignorer les alertes

In [ ]:
df = pd.read_csv("Churn_Modelling.csv") #importer la base de donn?e
df.shape

In [ ]:
df.head()

In [ ]:
df = df.drop(['RowNumber','CustomerId','Surname'], axis=1)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
df.isnull().sum()

In [ ]:
pd.crosstab(df.Exited, 'freq').plot.bar()

In [ ]:
pd.crosstab(df.Exited, 'freq').plot.pie(subplots=True, figsize = (4, 4))

In [ ]:
list_plot = ['Balance','EstimatedSalary']

sns.set_theme(style='whitegrid')
sns.boxplot(data=df[list_plot], orient='h')
plt.show()

In [ ]:
sns.set_theme(style='whitegrid')
sns.boxplot(df['CreditScore'],orient='h')
plt.show()

In [ ]:
q1 = df['CreditScore'].quantile(0.25)
q2 = df['CreditScore'].quantile(0.75)
IQR = q2-q1
b_inf = q1-1.5*IQR
b_sup = q2+1.5*IQR
df = df[df['CreditScore']>b_inf]
df = df[df['CreditScore']<b_sup]

In [ ]:
sns.set_theme(style='whitegrid')
sns.boxplot(df['CreditScore'],orient='h')
plt.show()

In [ ]:
sns.set_theme(style='whitegrid')
sns.boxplot(df['NumOfProducts'],orient='h')
plt.show()

In [ ]:
q1 = df['NumOfProducts'].quantile(0.25)
q2 = df['NumOfProducts'].quantile(0.75)
IQR = q2-q1
b_inf = q1-1.5*IQR
b_sup = q2+1.5*IQR
df = df[df['NumOfProducts']>b_inf]
df = df[df['NumOfProducts']<b_sup]

In [ ]:
sns.set_theme(style='whitegrid')
sns.boxplot(df['NumOfProducts'],orient='h')
plt.show()

In [ ]:
X = df.iloc[:, :10].values
y = df.iloc[:, 10].values

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
ct=ColumnTransformer([('state',OneHotEncoder(),[1, 2])],remainder='passthrough')
X=ct.fit_transform(x)

In [ ]:
X[0:5,:]

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)
X_train.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(X_train, y_train)

In [ ]:
lr.coef_

In [ ]:
lr.intercept_

In [ ]:
lr.coef_[0]

In [ ]:
y_pred = lr.predict(X_train)

for i in range(10):
    print([y_train[i],y_pred[i]])

In [ ]:
from sklearn import metrics

fpr, tpr, thr = metrics.roc_curve(y_train, y_pred)
roc_auc_log_reg = metrics.auc(fpr, tpr)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier()
model.fit(X_train, y_train)

In [ ]:
y_pred_prob = model.predict_proba(X_test)
y_pred = model.predict(X_test)
print(y_pred)

In [ ]:
print(y_pred[0])

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
fpr, tpr, thr = metrics.roc_curve(y_test, y_pred_prob[:, 1])
roc_auc_log_reg = metrics.auc(fpr, tpr)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier()
model.fit(X_train, y_train)

In [ ]:
y_pred_knn_proba = model.predict_proba(X_test)
y_pred_knn= model.predict(X_test)

print(y_pred_knn)

In [ ]:
from sklearn.model_selection import GridSearchCV

parameters = {'n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15]}
model = KNeighborsClassifier()
grid = GridSearchCV(model, parameters, scoring='accuracy', cv=5)
grid.fit(X_train, y_train)
print(grid.best_params_)
print(grid.best_score_)

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred_knn)
print(cm)

In [ ]:
fpr_knn, tpr_knn, _ = metrics.roc_curve(y_test, y_pred_knn_proba[:, 1])
roc_auc_knn = metrics.auc(fpr_knn, tpr_knn)

In [ ]:
plt.figure()
plt.plot(fpr_knn, tpr_knn, color='navy', lw=2, label='K-Nearest Neighbors (area = %0.2f)' % roc_auc_knn)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

In [ ]:
plt.plot(fpr_log_reg, tpr_log_reg, color='darkorange', lw=2, label='Logistic Regression (area = %0.2f)' % roc_auc_log_reg)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
